In [1]:
# CELL 1
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

engine = create_engine(
    "mssql+pyodbc://@LAPTOP-K44AI2HU\\SQLEXPRESS/pharma_db"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

In [2]:
# CELL 2 — Pull drug shortage statistics
# SQL Server uses DATEDIFF instead of date subtraction
drug_stats = pd.read_sql("""
    SELECT
        d.drug_id,
        d.generic_name,
        d.is_essential,
        d.criticality_tier,
        COUNT(*) as shortage_count,
        AVG(CAST(
            CASE
                WHEN f.shortage_end_date IS NOT NULL
                THEN DATEDIFF(day, f.shortage_start_date, f.shortage_end_date)
                ELSE DATEDIFF(day, f.shortage_start_date, CAST(GETDATE() AS DATE))
            END
        AS FLOAT)) as avg_duration_days,
        SUM(CASE WHEN f.status = 'ACTIVE' THEN 1 ELSE 0 END) as active_count,
        COUNT(DISTINCT f.manufacturer_id) as supplier_count
    FROM core.dim_drug d
    JOIN core.fact_shortage f ON f.drug_id = d.drug_id
    GROUP BY d.drug_id, d.generic_name, d.is_essential, d.criticality_tier
""", engine)

print(f"Drugs with shortage history: {len(drug_stats)}")
drug_stats.head()

Drugs with shortage history: 247


,drug_id,generic_name,is_essential,criticality_tier,shortage_count,avg_duration_days,active_count,supplier_count
0,1765,QUINAPRIL HYDROCHLORIDE TABLET,False,MEDIUM,24,1017.166667,20,5
1,1766,EVEROLIMUS TABLET,False,MEDIUM,11,0.000000,0,3
2,1767,PILOCARPINE HYDROCHLORIDE TABLET,False,MEDIUM,2,0.000000,0,1
3,1768,ETOMIDATE INJECTION,False,MEDIUM,19,1315.315789,19,6
4,1769,OMEPRAZOLE DELAYED RELEASE CAPSULE,False,MEDIUM,10,0.000000,0,1


In [3]:
# CELL 3 — Compute risk score (same logic as before)
def compute_risk(df):
    df = df.copy().fillna(0)

    def minmax(series):
        mn, mx = series.min(), series.max()
        if mx == mn: return pd.Series([50]*len(series), index=series.index)
        return ((series - mn) / (mx - mn)) * 100

    df["freq_score"]     = minmax(df["shortage_count"])
    df["duration_score"] = minmax(df["avg_duration_days"])
    df["active_score"]   = minmax(df["active_count"])
    df["sole_supplier"]  = (df["supplier_count"] == 1).astype(int) * 100

    tier_map = {"CRITICAL": 100, "HIGH": 75, "MEDIUM": 50, "LOW": 25}
    df["tier_score"] = df["criticality_tier"].map(tier_map).fillna(50)
    df["essential_score"] = df["is_essential"].astype(int) * 100

    df["risk_score"] = (
        df["freq_score"]     * 0.25 +
        df["duration_score"] * 0.20 +
        df["active_score"]   * 0.15 +
        df["sole_supplier"]  * 0.15 +
        df["tier_score"]     * 0.15 +
        df["essential_score"]* 0.10
    ).round(2)

    df["criticality_tier"] = pd.cut(
        df["risk_score"],
        bins=[0, 25, 50, 75, 100],
        labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
        include_lowest=True
    )
    return df

scored = compute_risk(drug_stats)
print("Risk score distribution:")
print(scored["criticality_tier"].value_counts())

Risk score distribution:
criticality_tier
LOW         207
MEDIUM       38
HIGH          2
CRITICAL      0
Name: count, dtype: int64


In [4]:
# CELL 4 — Update SQL Server database
with engine.begin() as conn:   # engine.begin() auto-commits in SQL Server
    for _, row in scored.iterrows():
        conn.execute(text("""
            UPDATE core.dim_drug
            SET risk_score = :score, criticality_tier = :tier
            WHERE drug_id = :id
        """), {"score": float(row["risk_score"]),
               "tier": str(row["criticality_tier"]),
               "id": int(row["drug_id"])})

print("✅ Risk scores updated in SQL Server!")
print(scored[["generic_name","risk_score","criticality_tier"]].sort_values("risk_score", ascending=False).head(10))

✅ Risk scores updated in SQL Server!
                                          generic_name  risk_score  \
183                  LIDOCAINE HYDROCHLORIDE INJECTION       55.01   
30                 LISDEXAMFETAMINE DIMESYLATE CAPSULE       52.29   
6    AMPHETAMINE ASPARTATE MONOHYDRATE, AMPHETAMINE...       46.07   
205                BUPIVACAINE HYDROCHLORIDE INJECTION       41.44   
20                 ROPIVACAINE HYDROCHLORIDE INJECTION       41.33   
54             DEXMEDETOMIDINE HYDROCHLORIDE INJECTION       40.91   
18   EPINEPHRINE BITARTRATE, LIDOCAINE HYDROCHLORID...       37.07   
125           ECHOTHIOPHATE IODIDE OPHTHALMIC SOLUTION       34.03   
5                HYDROMORPHONE HYDROCHLORIDE INJECTION       33.47   
225                    RIFAPENTINE TABLET, FILM COATED       33.07   

    criticality_tier  
183             HIGH  
30              HIGH  
6             MEDIUM  
205           MEDIUM  
20            MEDIUM  
54            MEDIUM  
18            MEDIUM  
125     